In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

import os
import pandas as pd 

os.getcwd()


In [ ]:

sourceDir = '/share/data/cellprofiler/automation/results'
rootDir = str(data_dir("exp1_main")) + "/"


### Define Functions


In [ ]:
metaEx = pd.read_csv(f'{rootDir}spher_colo52-metadata.csv')
these_cols = ['barcode','well_id', 'cmpdname', 'solvent', 'cmpd_conc', 'target', 'pathway', 'pubchemID', 'inkey', 'cell_line', 'article_id']
metaEx = metaEx[these_cols]

# Rename a few columns
metaEx = metaEx.rename(columns={'cmpd_conc': 'cmpd_conc_um', 'article_id': 'selleckchem_id'})

metaEx.head()

In [ ]:
Template = pd.read_csv(f'spher-colo52.tsv', sep='\t')

In [ ]:
Template.head()

# Add a new column wiht the barcode (starts with PB0001..)
Template['barcode'] = Template['Files'].str.slice(13, 21)

# # Regex to extract from the filename
Template[['well_id', 'z_plane', 'channel']] = Template['Files'].str.extract(r'Well-([A-Z0-9]{3})-z([0-9]{1,2})-(.*).ome.tiff')

Template.head()



In [ ]:
# Merge the two dataframes on barcode and well_id
FileList = pd.merge(Template, metaEx, on=['barcode', 'well_id'], how='left')

FileList.head()



In [ ]:
# Save the final metadata file
FileList.to_csv(f'spher_colo52-FileList.tsv', sep= '\t',index=False)

#### Now prepare the annotations file

In [ ]:
Annotations = pd.read_csv(f'results.tsv', sep='\t')

In [ ]:
Annotations

# # Regex to extract from the filename
Annotations[['barcode','Annotation Type','compartment','well_id', 'z_plane']] = Annotations['Files'].str.extract(r'results\/([A-Z0-9]{8})\/([a-z]{12})\/(.*)_.*_([A-Z0-9]{3})_([0-9]{1,2}).npy')

Annotations.loc[Annotations['compartment'] == 'cell', 'channel'] = 'PHAandWGA'
Annotations.loc[Annotations['compartment'] == 'nuclei', 'channel'] = 'HOECHST'

In [ ]:
# Replace column "Annotation Type" values "segmentation" with "Segmentation masks"
Annotations['Annotation Type'] = Annotations['Annotation Type'].replace(
    {'segmentation': 'Segmentation masks'})

Annotations


In [ ]:
Template.rename(columns={'Files': 'source image'}, inplace=True)

Template

In [ ]:
# Merge the two dataframes on barcode and well_id
Annotations_result = pd.merge(Annotations, Template, on=['barcode', 'well_id', 'z_plane', 'channel'], how='left')

Annotations_result.head()

In [ ]:
# Save the final metadata file
Annotations_result.to_csv(f'spher_colo52-Annotations.tsv', sep= '\t',index=False)